In [2]:
#importing libraries

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from lightgbm import LGBMRegressor
from sklearn.preprocessing import LabelEncoder
import plotly.express as px

In [3]:
#import data

data_europe = pd.read_csv('processed_data/data_europe.csv')
data_america = pd.read_csv('processed_data/data_america.csv')
data_australia = pd.read_csv('processed_data/data_australia.csv')

In [4]:
#verify columns

data_europe.columns

Index(['id', 'name', 'description', 'host_id', 'host_name', 'host_since',
       'host_acceptance_rate', 'host_is_superhost', 'host_listings_count',
       'host_total_listings_count', 'host_verifications',
       'host_has_profile_pic', 'host_identity_verified', 'latitude',
       'longitude', 'property_type', 'room_type', 'accommodates', 'bathrooms',
       'bedrooms', 'beds', 'amenities', 'price', 'has_availability',
       'availability_30', 'availability_60', 'availability_90',
       'availability_365', 'number_of_reviews', 'number_of_reviews_ltm',
       'number_of_reviews_l30d', 'city'],
      dtype='str')

In [5]:
#verify columns

data_europe.info()

<class 'pandas.DataFrame'>
RangeIndex: 104818 entries, 0 to 104817
Data columns (total 32 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   id                         104818 non-null  int64  
 1   name                       104818 non-null  str    
 2   description                104818 non-null  str    
 3   host_id                    104818 non-null  int64  
 4   host_name                  104818 non-null  str    
 5   host_since                 104818 non-null  str    
 6   host_acceptance_rate       104818 non-null  str    
 7   host_is_superhost          104818 non-null  str    
 8   host_listings_count        104818 non-null  float64
 9   host_total_listings_count  104818 non-null  float64
 10  host_verifications         104818 non-null  str    
 11  host_has_profile_pic       104818 non-null  str    
 12  host_identity_verified     104818 non-null  str    
 13  latitude                   104818 non-nu

In [6]:
#verify coluns

data_europe.head()

,id,name,description,host_id,host_name,host_since,host_acceptance_rate,host_is_superhost,host_listings_count,host_total_listings_count,...,price,has_availability,availability_30,availability_60,availability_90,availability_365,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,city
0,11616,Habitación 5 -3,Enjoy the simplicity of this quiet and central...,18191427,David,2014-07-16,97%,f,27.0,28.0,...,60.0,t,26,50,75,347,5,5,3,Barcelona
1,89610,Spacious 3 bedroom Apartment,A spacious and beautiful 3 bedroom & 2 bathroo...,36818617,My London,2015-06-26,100%,f,70.0,111.0,...,204.0,t,9,39,69,69,1,1,0,London
2,253662,Stile Veneziano Centrale,Nice quiet room,501069099,Bgroup Srl,2023-02-14,100%,f,14.0,14.0,...,87.0,t,19,38,63,334,5,0,0,Venice
3,21758,Loue 1 chambre,"Accommodation in a 3-story residence, close to...",651783519,Brigitte,2024-09-13,56%,f,1.0,1.0,...,32.0,t,14,44,74,349,4,4,1,Bordeaux
4,144090,90m2 für bis zu 12 Personen - München Innenstadt,Enjoy freedom and flexibility. Homaris Apartme...,446752152,Flo,2022-02-25,100%,f,6.0,8.0,...,186.0,t,11,23,39,271,30,29,3,Munich


In [7]:
#remove string columns

features = data_europe.select_dtypes(include='number')



In [8]:
#add coluns room type and property_type

features ['room_type'] = data_europe['room_type']
features ['property_type'] = data_europe['property_type']


In [9]:
#checking columns

features.head(20)

,id,host_id,host_listings_count,host_total_listings_count,latitude,longitude,accommodates,bathrooms,bedrooms,beds,price,availability_30,availability_60,availability_90,availability_365,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,room_type,property_type
0,11616,18191427,27.0,28.0,41.391680,2.161700,2,2.0,1.0,1.0,60.0,26,50,75,347,5,5,3,Private room,Private room in rental unit
1,89610,36818617,70.0,111.0,51.546420,-0.220100,7,2.0,3.0,4.0,204.0,9,39,69,69,1,1,0,Entire home/apt,Entire serviced apartment
2,253662,501069099,14.0,14.0,45.445515,12.321438,2,1.0,1.0,1.0,87.0,19,38,63,334,5,0,0,Private room,Private room in rental unit
3,21758,651783519,1.0,1.0,44.889460,-0.644700,2,1.0,1.0,1.0,32.0,14,44,74,349,4,4,1,Private room,Private room in rental unit
4,144090,446752152,6.0,8.0,48.124752,11.556081,12,1.0,2.0,6.0,186.0,11,23,39,271,30,29,3,Entire home/apt,Entire rental unit
5,224302,9483466,2.0,2.0,37.572730,12.948100,6,2.0,3.0,3.0,219.0,0,0,29,280,3,3,0,Entire home/apt,Entire villa
6,41332,19577782,8.0,10.0,35.328950,25.204232,3,1.0,1.0,1.0,80.0,30,60,90,365,3,1,0,Private room,Private room in rental unit
7,189862,552119334,1.0,1.0,40.073122,18.168747,5,1.0,2.0,4.0,57.0,21,51,81,198,0,0,0,Entire home/apt,Entire home
8,87038,479535561,40.0,48.0,51.486272,-0.168413,8,4.0,4.0,4.0,800.0,0,28,58,333,0,0,0,Entire home/apt,Entire rental unit
9,64332,18054334,4.0,4.0,53.466780,-2.285990,2,1.5,0.0,1.0,51.0,20,50,80,289,18,5,1,Entire home/apt,Entire condo


In [10]:
#encoding str columns

encoder = LabelEncoder()

features['room_type'] = encoder.fit_transform(features['room_type'])

features['property_type'] = encoder.fit_transform(features['property_type'])

features = features.drop(columns= ['host_id', 'availability_365','availability_90', 'availability_60', 'availability_30', 'host_listings_count', 'id'] )

In [11]:
features.head()

,host_total_listings_count,latitude,longitude,accommodates,bathrooms,bedrooms,beds,price,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,room_type,property_type
0,28.0,41.391680,2.161700,2,2.0,1.0,1.0,60.0,5,5,3,2,66
1,111.0,51.546420,-0.220100,7,2.0,3.0,4.0,204.0,1,1,0,0,25
2,14.0,45.445515,12.321438,2,1.0,1.0,1.0,87.0,5,0,0,2,66
3,1.0,44.889460,-0.644700,2,1.0,1.0,1.0,32.0,4,4,1,2,66
4,8.0,48.124752,11.556081,12,1.0,2.0,6.0,186.0,30,29,3,0,24


In [12]:
#data split

X = features.drop(columns='price')
y = features['price']

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=1
)

In [13]:
LGBM = LGBMRegressor(
    objective='poisson',
    n_estimators=500,
    learning_rate=0.2,
    num_leaves=64,
    max_depth=8,
    min_child_samples=50,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=1
)

LGBM.fit(X_train, y_train)
y_pred = LGBM.predict(X_valid)

#evaluating model

mae = mean_absolute_error(y_valid, y_pred)
rmse = np.sqrt(mean_squared_error(y_valid, y_pred))
r2 = r2_score(y_valid, y_pred)


print ('MAE', mae,'RMSE', rmse,'R2', r2)



[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006412 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1369
[LightGBM] [Info] Number of data points in the train set: 83854, number of used features: 12
[LightGBM] [Info] Start training from score 5.079709
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain:

In [14]:
feature_importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': LGBM.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by='Importance',
    ascending=False
)

feature_importance.head(20)

,Feature,Importance
2,longitude,5149
1,latitude,4947
0,host_total_listings_count,4451
7,number_of_reviews,3073
8,number_of_reviews_ltm,2070
11,property_type,1852
3,accommodates,1436
6,beds,1207
4,bathrooms,1118
5,bedrooms,919


Now applying the same idea to American

In [15]:
#remove string columns

feature_AME = data_america.select_dtypes(include='number')

#add columns room type and property_type

feature_AME['room_type'] = data_america['room_type']
feature_AME['property_type'] = data_america['property_type']

#encoding str columns

encoder = LabelEncoder()

feature_AME ['room_type'] = encoder.fit_transform(feature_AME['room_type'])

feature_AME['property_type'] = encoder.fit_transform(feature_AME['property_type'])

feature_AME = feature_AME.drop(
    columns=['host_id', 'availability_365', 'availability_90', 'availability_60', 'availability_30',
             'host_listings_count', 'id'])

#data split

X_AME = feature_AME.drop(columns='price')
y_AME= feature_AME['price']

X_train_AME, X_valid_AME, y_train_AME, y_valid_AME = train_test_split(
    X_AME,
    y_AME,
    test_size=0.2,
    random_state=1
)
LGBM_AME = LGBMRegressor(
                     objective='poisson',
                     n_estimators=500,
                     learning_rate=0.2,
                     num_leaves=64,
                     max_depth=8,
                     min_child_samples=50,
                     subsample=0.8,
                     colsample_bytree=0.8,
                     random_state=1
                     )

LGBM_AME.fit(X_train_AME, y_train_AME)
y_pred_AME = LGBM_AME.predict(X_valid_AME)

#evaluating model

mae = mean_absolute_error(y_valid_AME, y_pred_AME)
rmse = np.sqrt(mean_squared_error(y_valid_AME, y_pred_AME))
r2 = r2_score(y_valid_AME, y_pred_AME)

print('MAE', mae, 'RMSE', rmse, 'R2', r2)



[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.005060 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1313
[LightGBM] [Info] Number of data points in the train set: 37398, number of used features: 12
[LightGBM] [Info] Start training from score 5.351035
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain,

In [16]:
#remove string columns

feature_AUS = data_australia.select_dtypes(include='number')

#add coluns room type and property_type

feature_AUS['room_type'] = data_australia['room_type']
feature_AUS['property_type'] = data_australia['property_type']

#encoding str columns

encoder = LabelEncoder()

feature_AUS ['room_type'] = encoder.fit_transform(feature_AUS['room_type'])

feature_AUS['property_type'] = encoder.fit_transform(feature_AUS['property_type'])

feature_AUS = feature_AUS.drop(
    columns=['host_id', 'availability_365', 'availability_90', 'availability_60', 'availability_30',
             'host_listings_count', 'id'])

#data split

X_AUS = feature_AUS.drop(columns='price')
y_AUS= feature_AUS['price']

X_train_AUS, X_valid_AUS, y_train_AUS, y_valid_AUS = train_test_split(
    X_AUS,
    y_AUS,
    test_size=0.2,
    random_state=1
)
LGBM_AUS = LGBMRegressor(
                     objective='poisson',
                     n_estimators=500,
                     learning_rate=0.2,
                     num_leaves=64,
                     max_depth=8,
                     min_child_samples=50,
                     subsample=0.8,
                     colsample_bytree=0.8,
                     random_state=1
                     )

LGBM_AUS.fit(X_train_AUS, y_train_AUS)
y_pred_AUS = LGBM_AUS.predict(X_valid_AUS)

#evaluating model

mae = mean_absolute_error(y_valid_AUS, y_pred_AUS)
rmse = np.sqrt(mean_squared_error(y_valid_AUS, y_pred_AUS))
r2 = r2_score(y_valid_AUS, y_pred_AUS)

print('MAE', mae, 'RMSE', rmse, 'R2', r2)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000972 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1138
[LightGBM] [Info] Number of data points in the train set: 7006, number of used features: 12
[LightGBM] [Info] Start training from score 5.648996
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: 

In [17]:
#plot expected vs found AUS

#criate dataframe

PredXreal = pd.DataFrame({
    'pred_y_AUS': y_pred_AUS,
    'y_valid_AUS': y_valid_AUS,
})

fig = px.scatter(PredXreal, x='pred_y_AUS', y='y_valid_AUS')

In [18]:
rmse = np.sqrt(mean_squared_error(y_valid_AUS, y_pred_AUS))
r2 = r2_score(y_valid_AUS, y_pred_AUS)

fig.update_layout(
    annotations=[
        dict(
            x=0.05,
            y=0.95,
            xref='paper',
            yref='paper',
            text=f'RMSE: {rmse:.2f}<br>R²: {r2:.3f}',
            showarrow=False,
            font=dict(size=12)
        )
    ]
)

fig.show()

In [19]:
PredXreal = PredXreal.sort_values(by = 'pred_y_AUS', ascending = True)

In [20]:
PredXreal.head()

,pred_y_AUS,y_valid_AUS
1140,39.678283,43.0
1338,46.033799,46.0
8182,46.357294,60.0
5291,47.368167,47.0
4281,47.652391,45.0


In [21]:
data_australia.info()

<class 'pandas.DataFrame'>
RangeIndex: 8758 entries, 0 to 8757
Data columns (total 32 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   id                         8758 non-null   int64  
 1   name                       8758 non-null   str    
 2   description                8758 non-null   str    
 3   host_id                    8758 non-null   int64  
 4   host_name                  8758 non-null   str    
 5   host_since                 8758 non-null   str    
 6   host_acceptance_rate       8758 non-null   str    
 7   host_is_superhost          8758 non-null   str    
 8   host_listings_count        8758 non-null   float64
 9   host_total_listings_count  8758 non-null   float64
 10  host_verifications         8758 non-null   str    
 11  host_has_profile_pic       8758 non-null   str    
 12  host_identity_verified     8758 non-null   str    
 13  latitude                   8758 non-null   float64
 14  lon

In [22]:
PredXreal.info()

<class 'pandas.DataFrame'>
Index: 1752 entries, 1140 to 6129
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   pred_y_AUS   1752 non-null   float64
 1   y_valid_AUS  1752 non-null   float64
dtypes: float64(2)
memory usage: 41.1 KB
